# Lab 16: Steady-State Analysis

**Problem Statement:**
Analyze the warm-up period of a non-terminating simulation (like a continuous manufacturing line) and calculate the steady-state mean time.


### Theory and Approach

In a **non-terminating simulation** (one that runs indefinitely), the system often starts empty and idle. This initial state is unrepresentative of the system's long-term typical behavior. 

- **Warm-up Period**: The initial transient phase where the system is approaching its typical operating conditions.
- **Steady-State**: The phase where the probability distributions of the state variables become independent of time.

**Welch's Method for Warm-up Analysis**:
1. Make $n$ independent replications of the simulation, each of length $m$.
2. Let $Y_{ji}$ be the response of the $i$-th observation in the $j$-th replication.
3. Calculate the ensemble average across replications: $\\bar{Y}_i = \\frac{1}{n} \sum_{j=1}^n Y_{ji}$
4. Plot the moving average of $\\bar{Y}_i$ to smooth out high-frequency noise and visually identify where the curve flattens out. This point is the end of the warm-up period ($l$).
5. The **steady-state mean** is calculated by discarding the first $l$ observations of each replication.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)

# ==========================================
# 1. Generate Mock Non-Terminating Data
# ==========================================
# We'll mock the 'queue length' of a manufacturing line.
# It starts at 0, grows, and eventually fluctuates around a steady mean of 25.
n_replications = 10
m_observations = 500

replications_data = []

for _ in range(n_replications):
    # Simulating a transient phase that approaches 25
    time = np.arange(m_observations)
    # y = 25 - 25*exp(-0.02*t) + noise
    transient = 25 * (1 - np.exp(-0.02 * time))
    noise = np.random.normal(0, 3, m_observations)
    obs = np.maximum(0, transient + noise) # Queue length can't be negative
    replications_data.append(obs)
    
replications_data = np.array(replications_data)

# ==========================================
# 2. Welch's Method: Ensemble Averages
# ==========================================
ensemble_averages = np.mean(replications_data, axis=0)

# Moving Average to smooth the curve (window size = 20)
w = 20
moving_averages = []
for i in range(1, m_observations + 1):
    if i <= w:
        ma = np.mean(ensemble_averages[:(2*i-1)])
    else:
        ma = np.mean(ensemble_averages[(i-w):(i+w)])
    moving_averages.append(ma)

# ==========================================
# 3. Identify Warm-up and Calculate Steady-State
# ==========================================
# Visually, the curve seems to flatten around observation 150
warmup_period = 150

# Calculate steady-state mean by discarding the warmup period
# We flatten the remaining data from all replications
steady_state_data = replications_data[:, warmup_period:].flatten()
steady_state_mean = np.mean(steady_state_data)

print(f"Identified Warm-up Period (l): {warmup_period} observations")
print(f"Calculated Steady-State Mean : {steady_state_mean:.2f}")

# ==========================================
# 4. Visualization
# ==========================================
plt.figure(figsize=(12, 6))

# Plot one replication to show noise
plt.plot(replications_data[0], alpha=0.3, color='gray', label='Single Replication')

# Plot moving average
plt.plot(moving_averages, color='blue', linewidth=2, label='Smoothed Ensemble Average')

# Mark Warmup Period
plt.axvline(warmup_period, color='red', linestyle='--', linewidth=2, label=f'End of Warm-up (l={warmup_period})')

# Mark Steady State Mean
plt.axhline(steady_state_mean, color='green', linestyle=':', linewidth=2, label=f'Steady-State Mean ({steady_state_mean:.2f})')

plt.title("Welch's Method for Warm-up Period Analysis", fontsize=14, fontweight='bold')
plt.xlabel("Observation Number (Time)", fontsize=12)
plt.ylabel("System Response (e.g., Queue Length)", fontsize=12)
plt.legend()
plt.grid(alpha=0.4)
plt.gca().spines['top'].set_visible(False)
plt.gca().spines['right'].set_visible(False)
plt.tight_layout()
plt.show()
